# Donut 학습용 데이터 라벨링 파이프라인

도면 요소(measure / radii / gdt / titleblock)를 **Donut 학습 포맷(이미지+JSON)** 으로 만든다.

```
[PDF/이미지] → (YOLO-OBB 검출·deskew 크롭 | 미리 자른 크롭)
            → Qwen 자동라벨(부트스트랩) → Donut 포맷 + 분할 + 검수 manifest
            → donut_vml 로 변환·학습
```

> 이 노트북은 `donut_data/*.py` 와 동일 로직의 실행형 버전이다(커널: **kardi_env**).
> 설계 근거: `resource/도면요소추출_방법비교.md` 0-A.

In [ ]:
# ── 공통 import ───────────────────────────────────────────────
import os, sys, json, csv, re, random, shutil
print("import 완료 ✓")

## 1. 스키마 · task_prompt · 자동라벨 프롬프트
클래스별 정답 JSON 형식과 Qwen 프롬프트를 정의한다.
클래스 인덱스/이름은 `drawing_obb.yaml`(0=measure 1=gdt 2=radii)과 일치시킬 것.

In [ ]:
# Donut 디코더의 시작 토큰 겸 태스크 식별자. 학습 시 tokenizer 에 special token 으로 추가.
TASK_PROMPT = {
    "measure":    "<s_measure>",
    "radii":      "<s_radii>",
    "gdt":        "<s_gdt>",
    "titleblock": "<s_titleblock>",
}

# 클래스별 빈 스키마(=필드 목록). --no-model(수작업) 모드에서 템플릿으로 사용.
EMPTY_SCHEMA = {
    "measure":    {"nominal": None, "upper": None, "lower": None, "raw": None},
    "radii":      {"nominal": None, "upper": None, "lower": None, "raw": None},
    "gdt":        {"characteristic": None, "symbol": None, "tolerance": None,
                   "modifier": None, "datums": [], "raw": None},
    "titleblock": {"Title": None, "Drawing No.": None, "LIC. Material": None,
                   "Material": None, "Rev": None},
}

CLASSES = list(EMPTY_SCHEMA.keys())

# ── Qwen 자동라벨 프롬프트(부트스트랩용). 출력은 strict JSON 한 개. ──────────────
_MEASURE_RULES = (
    "Extract the dimension/tolerance into STRICT JSON with keys exactly:\n"
    '  "nominal" - the nominal value as printed (e.g. "Ø36 H8", "78", "20")\n'
    '  "upper"   - upper tolerance as printed (e.g. "+0,3", "+0,1") or null\n'
    '  "lower"   - lower tolerance as printed (e.g. "-0,1", "0") or null\n'
    '  "raw"     - verbatim transcription of the whole patch\n'
    "Rules: copy VERBATIM; preserve European decimal commas (\"0,1\" stays \"0,1\"); "
    "keep \"Ø\" in nominal; if a field is absent use null. Output ONLY the JSON object."
)

PROMPTS = {
    "measure": _MEASURE_RULES,
    "radii": _MEASURE_RULES.replace(
        'the nominal value as printed (e.g. "Ø36 H8", "78", "20")',
        'the radius as printed (e.g. "R3", "R42")',
    ),
    "gdt": (
        "This crop is ONE GD&T (Geometric Dimensioning & Tolerancing) feature control frame.\n"
        "GD&T symbol -> characteristic (use the English name):\n"
        "  straightness, flatness(⏥/▱), circularity, cylindricity, profile_line, profile_surface,\n"
        "  perpendicularity(⊥), angularity, parallelism, position(⌖/⊕), concentricity, symmetry,\n"
        "  circular_runout, total_runout\n"
        "Modifiers (circled letter): Ⓜ=M(MMC), Ⓛ=L(LMC), Ⓢ=S(RFS), Ⓟ=P(projected).\n"
        "Return STRICT JSON with keys exactly:\n"
        '  "characteristic" - english name above, or null\n'
        '  "symbol"         - the glyph as printed (e.g. "⌖","⊥","▱")\n'
        '  "tolerance"      - zone value as printed (e.g. "Ø0.4","0.08")\n'
        '  "modifier"       - "M"|"L"|"S"|"P"|null (modifier on the tolerance value)\n'
        '  "datums"         - ordered list WITH modifiers, e.g. ["A","B(M)","C"]; [] if none\n'
        '  "raw"            - verbatim transcription, left to right\n'
        "Rules: copy VERBATIM; preserve European commas; a circled letter on a datum stays inside "
        'that entry (B+Ⓜ -> "B(M)"). Output ONLY the JSON object.'
    ),
    # 표제란 5필드 — extract_title_block.py 의 정의와 동일하게 유지.
    "titleblock": (
        "This image is the title block of a mechanical engineering drawing.\n"
        "Extract exactly these five fields as STRICT JSON with keys: "
        '"Title", "Drawing No.", "LIC. Material", "Material", "Rev".\n'
        '- "Rev": value at the BOTTOM of the narrow far-right "Rev" column (not the "Security" cell).\n'
        "- Copy text verbatim. If a field cannot be found, set it to null.\n"
        "- Output ONLY the JSON object."
    ),
}

# 알려진 재질 코드의 표준 표기(대소문자만 교정) — extract_title_block.py 와 동일.
CANONICAL_MATERIALS = ["SCr18N8", "SUS316L", "SCrNi"]

## 2. Qwen2.5-VL 자동 라벨러
모델을 한 번만 로드해 여러 크롭에 재사용한다. greedy 디코딩이라 결정적.
출력 JSON 은 `parse_json()` 으로 안전 파싱하고, 표제란 재질은 대소문자만 교정.

In [ ]:
import json
import os
import re



def parse_json(text: str):
    """모델 출력에서 첫 {...} JSON 객체만 안전 추출(코드펜스 제거). 실패 시 예외."""
    cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
    start, end = cleaned.find("{"), cleaned.rfind("}")
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return json.loads(cleaned)


def fix_material_case(value):
    """알려진 재질코드의 대소문자만 표준 표기로 교정(문자 자체는 불변)."""
    if not isinstance(value, str):
        return value
    key = value.strip().casefold()
    for canon in CANONICAL_MATERIALS:
        if canon.casefold() == key:
            return canon
    return value


class Labeler:
    """Qwen2.5-VL 라벨러. shots(few-shot 예시)는 클래스별 (이미지경로, 정답dict) 리스트."""

    def __init__(self, model_id="Qwen/Qwen2.5-VL-7B-Instruct", shots=None):
        import torch
        from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

        self.torch = torch
        self.shots = shots or {}
        print(f"[autolabel] loading {model_id} ...", flush=True)
        self.model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            model_id, torch_dtype="auto", device_map="auto")
        self.processor = AutoProcessor.from_pretrained(model_id)

    @staticmethod
    def _img(path):
        return {"type": "image", "image": f"file://{os.path.abspath(path)}"}

    def _messages(self, image_path, cls):
        prompt = PROMPTS[cls]
        msgs = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        for shot_img, shot_gold in self.shots.get(cls, []):       # few-shot(있으면)
            msgs.append({"role": "user",
                         "content": [self._img(shot_img), {"type": "text", "text": "Extract:"}]})
            msgs.append({"role": "assistant",
                         "content": [{"type": "text",
                                      "text": json.dumps(shot_gold, ensure_ascii=False)}]})
        msgs.append({"role": "user",
                     "content": [self._img(image_path), {"type": "text", "text": "Extract:"}]})
        return msgs

    def label(self, image_path, cls):
        """크롭 1장 -> (라벨dict, 원문, 성공여부). 파싱 실패 시 빈 스키마 + ok=False."""
        from qwen_vl_utils import process_vision_info

        messages = self._messages(image_path, cls)
        text = self.processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = self.processor(text=[text], images=image_inputs, videos=video_inputs,
                                padding=True, return_tensors="pt").to(self.model.device)
        with self.torch.no_grad():
            gen = self.model.generate(**inputs, max_new_tokens=256, do_sample=False)
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
        raw = self.processor.batch_decode(
            trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0].strip()

        try:
            result = parse_json(raw)
        except json.JSONDecodeError:
            return dict(EMPTY_SCHEMA[cls]), raw, False

        # 스키마 키 정렬 + 표제란 재질 대소문자 교정
        out = dict(EMPTY_SCHEMA[cls])
        for k in out:
            if k in result:
                out[k] = result[k]
        if cls == "titleblock":
            for f in ("LIC. Material", "Material"):
                out[f] = fix_material_case(out[f])
        return out, raw, True

## 3. (선택) 단일 크롭 자동라벨 테스트
7B 로드(수 분)가 필요하다. 빠르게 건너뛰려면 이 셀을 실행하지 말 것.

In [ ]:
# 7B 로드 + 1장 라벨 테스트 (선택)
# lab = Labeler()
# res, raw, ok = lab.label("input_doc/test_title_01.png", "titleblock")
# print(ok); print(json.dumps(res, ensure_ascii=False, indent=2))

## 4. 입력 수집 헬퍼
- `patches` 모드: 미리 자른 크롭 `<input>/<class>/*.png` (YOLO 불필요)
- `detect` 모드: 전체 도면(PDF/PNG) + 학습된 YOLO-OBB 가중치 (`ultralytics` 필요)

In [ ]:
import csv
import json
import os
import sys


IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")


# ── 입력 수집 ────────────────────────────────────────────────────────────────
def collect_patches(input_dir, classes):
    """patches 모드: <input>/<class>/*.{img} 를 (이미지경로, class) 리스트로."""
    items = []
    for cls in classes:
        d = os.path.join(input_dir, cls)
        if not os.path.isdir(d):
            print(f"  [skip] 폴더 없음: {d}")
            continue
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(IMG_EXTS):
                items.append((os.path.join(d, fn), cls))
    return items


def pdfs_to_images(input_dir, work_dir, dpi=300):
    """input_dir 의 PDF 를 work_dir 에 PNG 로 변환하고, PNG 경로 + 원래 이미지들을 합쳐 반환."""
    from pdf2image import convert_from_path

    os.makedirs(work_dir, exist_ok=True)
    pages = []
    for fn in sorted(os.listdir(input_dir)):
        path = os.path.join(input_dir, fn)
        stem, ext = os.path.splitext(fn)
        if ext.lower() == ".pdf":
            for i, img in enumerate(convert_from_path(path, dpi=dpi)):
                out = os.path.join(work_dir, f"{stem}_p{i}.png")
                img.save(out)
                pages.append(out)
        elif ext.lower() in IMG_EXTS:
            pages.append(path)
    return pages


def detect_and_crop(image_paths, weights, out_crop_dir, classes, conf=0.25, pad=0.06):
    """detect 모드: YOLO-OBB 로 검출 → minAreaRect deskew 크롭 → out_crop_dir/<class>/ 저장."""
    import cv2
    import numpy as np
    from ultralytics import YOLO

    model = YOLO(weights)
    names = model.names                       # {0:'measure',1:'gdt',2:'radii'}
    items = []
    for img_path in image_paths:
        img = cv2.imread(img_path)
        if img is None:
            print(f"  [warn] 읽기 실패: {img_path}");  continue
        res = model.predict(img_path, conf=conf, verbose=False)[0]
        if res.obb is None or len(res.obb) == 0:
            continue
        polys = res.obb.xyxyxyxy.cpu().numpy().reshape(-1, 4, 2)   # (N,4,2)
        clsids = res.obb.cls.cpu().numpy().astype(int)
        stem = os.path.splitext(os.path.basename(img_path))[0]
        for k, (poly, cid) in enumerate(zip(polys, clsids)):
            cls = names.get(int(cid), str(cid))
            if cls not in classes:
                continue
            crop = _deskew_crop(img, poly, cv2, np, pad)
            if crop is None or crop.size == 0:
                continue
            d = os.path.join(out_crop_dir, cls);  os.makedirs(d, exist_ok=True)
            out = os.path.join(d, f"{stem}_{k:03d}.png")
            cv2.imwrite(out, crop)
            items.append((out, cls))
    return items


def _deskew_crop(img, poly, cv2, np, pad):
    """OBB 4코너 → 회전 보정해 수평으로 펴서 크롭(약간 패딩)."""
    rect = cv2.minAreaRect(poly.astype(np.float32))     # ((cx,cy),(w,h),angle)
    (cx, cy), (w, h), angle = rect
    if w < 2 or h < 2:
        return None
    w2, h2 = int(w * (1 + pad)), int(h * (1 + pad))
    M = cv2.getRotationMatrix2D((cx, cy), angle, 1.0)
    rotated = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]),
                             flags=cv2.INTER_CUBIC, borderValue=(255, 255, 255))
    crop = cv2.getRectSubPix(rotated, (w2, h2), (cx, cy))
    if crop is not None and crop.shape[0] > crop.shape[1]:   # 세로형이면 가로로 회전
        crop = cv2.rotate(crop, cv2.ROTATE_90_CLOCKWISE)
    return crop


# ── 분할 ─────────────────────────────────────────────────────────────────────
def split_index(n_items, ratios, seed=42):
    """결정적 분할: 정렬된 인덱스를 비율대로 train/val/test 에 배정."""
    import random
    idx = list(range(n_items))
    random.Random(seed).shuffle(idx)
    n_tr = int(n_items * ratios[0])
    n_va = int(n_items * ratios[1])
    assign = {}
    for j, i in enumerate(idx):
        assign[i] = "train" if j < n_tr else ("val" if j < n_tr + n_va else "test")
    return assign


# ── 메인 ─────────────────────────────────────────────────────────────────────

In [ ]:
def _load_shots(shots_root, classes):
    """few-shot 예시 로드: shots/<class>/<name>.png + 같은이름.json"""
    shots = {}
    for cls in classes:
        d = os.path.join(shots_root, cls)
        if not os.path.isdir(d):
            continue
        pairs = []
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(IMG_EXTS):
                jp = os.path.join(d, os.path.splitext(fn)[0] + ".json")
                if os.path.isfile(jp):
                    with open(jp, encoding="utf-8") as fh:
                        pairs.append((os.path.join(d, fn), json.load(fh)))
        if pairs:
            shots[cls] = pairs
    return shots

## 5. 데이터셋 빌드 드라이버
수집 → (자동라벨) → Donut 포맷 + train/val/test 분할 + `manifest.csv`.

In [ ]:
def build_donut_dataset(mode, input_dir, out,
                        classes=("measure", "radii", "gdt"),
                        weights="yolo_obb_drawing/train/weights/best.pt", dpi=300,
                        split=(0.7, 0.15, 0.15),
                        model_id="Qwen/Qwen2.5-VL-7B-Instruct",
                        shots_root=None, no_model=False):
    """수집 → (자동라벨) → Donut 포맷(images/+labels/) + 분할 + manifest.csv."""
    # 1) 입력 크롭 수집
    if mode == "patches":
        items = collect_patches(input_dir, list(classes))
    else:
        pages = pdfs_to_images(input_dir, os.path.join(out, "_pages"), dpi=dpi)
        print(f"[detect] 페이지 {len(pages)}장 → YOLO-OBB 검출/크롭 ...")
        items = detect_and_crop(pages, weights, os.path.join(out, "_crops"), list(classes))
    if not items:
        raise SystemExit("크롭이 없습니다. 입력 경로/클래스 폴더를 확인하세요.")
    print(f"[수집] 크롭 {len(items)}개")

    # 2) (선택) few-shot 예시
    shots = _load_shots(shots_root, list(classes)) if shots_root else None

    # 3) 라벨러
    labeler = None if no_model else Labeler(model_id=model_id, shots=shots)

    # 4) 분할 + 디렉토리
    assign = split_index(len(items), split)
    for sp in ("train", "val", "test"):
        os.makedirs(os.path.join(out, sp, "images"), exist_ok=True)
        os.makedirs(os.path.join(out, sp, "labels"), exist_ok=True)

    # 5) 라벨 + 저장 + 매니페스트
    rows, n_ok, n_fail = [], 0, 0
    for i, (img_path, cls) in enumerate(items):
        sp = assign[i]
        uid = f"{cls}_{i:05d}"
        ext = os.path.splitext(img_path)[1].lower()
        dst_img = os.path.join(out, sp, "images", uid + ext)
        dst_lbl = os.path.join(out, sp, "labels", uid + ".json")
        shutil.copy(img_path, dst_img)

        if labeler is None:
            label, raw, ok = dict(EMPTY_SCHEMA[cls]), "", False
        else:
            label, raw, ok = labeler.label(img_path, cls)
        n_ok += int(ok); n_fail += int(not ok)

        out_obj = {"task": TASK_PROMPT[cls], **label}
        with open(dst_lbl, "w", encoding="utf-8") as fh:
            json.dump(out_obj, fh, ensure_ascii=False, indent=2)

        rows.append({
            "id": uid, "class": cls, "split": sp, "src": img_path,
            "autolabeled": int(labeler is not None), "parse_ok": int(ok),
            "verified": 0, "raw": (raw[:120].replace("\n", " ") if raw else ""),
        })

    with open(os.path.join(out, "manifest.csv"), "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=list(rows[0].keys()))
        w.writeheader(); w.writerows(rows)

    by = {sp: sum(1 for r in rows if r["split"] == sp) for sp in ("train", "val", "test")}
    print(f"\n[완료] {out}")
    print(f"  크롭 {len(items)}  train/val/test = {by['train']}/{by['val']}/{by['test']}")
    if labeler is not None:
        print(f"  자동라벨 파싱 성공 {n_ok} / 실패 {n_fail}")
    print(f"  검수: {os.path.join(out, 'manifest.csv')} (verified=1 로 표시)")
    return out

### 5-1. 실행 — 설정 후 호출
콜드스타트: 먼저 `patches/<class>/` 에 크롭 몇 장을 두고 아래를 실행.
처음엔 `no_model=True` 로 구조만 확인하고, 이후 `False` 로 Qwen 자동라벨.

In [ ]:
# ── 설정 ──────────────────────────────────────────────
MODE       = "patches"            # "patches" | "detect"
INPUT_DIR  = "patches"            # patches: 크롭 루트 / detect: 도면 폴더
OUT_DIR    = "datasets/donut_anno"
CLASSES    = ["measure", "radii", "gdt"]   # 필요시 "titleblock" 추가
WEIGHTS    = "yolo_obb_drawing/train/weights/best.pt"   # detect 모드
NO_MODEL   = True                 # True=빈 템플릿만 / False=Qwen 자동라벨
SHOTS_ROOT = None                 # few-shot 예시 루트(선택)

build_donut_dataset(MODE, INPUT_DIR, OUT_DIR, classes=CLASSES,
                    weights=WEIGHTS, no_model=NO_MODEL, shots_root=SHOTS_ROOT)

## 6. donut_vml 로 변환 (학습 연결)
donut_vml 의 `DonutDataset` 는 라벨 전체를 `json2token` 하므로 보조키 `task` 를 제거하고,
`build_model_and_processor` 가 task_prompt 토큰만 등록하므로 **클래스별 변환**이 안전하다.

In [ ]:
import csv
import json
import os
import shutil
import sys


IMG_EXTS = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

# 클래스별 권장 학습 설정(패치 특성 반영). 노트북 CFG 에 반영하기 위한 힌트.
#   - 주석 크롭(measure/radii/gdt)은 짧고 가로로 김 → max_length 작게, 해상도 중간.
#   - 표제란은 더 큰 영역 → 해상도 크게.
RECO = {
    "measure":    {"image_size": [320, 768], "max_length": 96},
    "radii":      {"image_size": [320, 768], "max_length": 64},
    "gdt":        {"image_size": [320, 960], "max_length": 128},
    "titleblock": {"image_size": [960, 768], "max_length": 160},
}


def load_verified(src):
    """manifest.csv → verified==1 인 id 집합. 없으면 None(=전체 허용)."""
    mp = os.path.join(src, "manifest.csv")
    if not os.path.isfile(mp):
        return None
    ok = set()
    with open(mp, encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            if str(row.get("verified", "0")).strip() == "1":
                ok.add(row["id"])
    return ok


def find_image(img_dir, stem):
    for ext in IMG_EXTS:
        p = os.path.join(img_dir, stem + ext)
        if os.path.isfile(p):
            return p
    return None


def convert_split(src, split, cls, dst_root, verified):
    """src/<split> → dst_root/<split> 으로 해당 클래스·검수통과 샘플 복사. 복사 수 반환."""
    in_lbl = os.path.join(src, split, "labels")
    in_img = os.path.join(src, split, "images")
    if not os.path.isdir(in_lbl):
        return 0
    out_img = os.path.join(dst_root, split, "images")
    out_lbl = os.path.join(dst_root, split, "labels")
    os.makedirs(out_img, exist_ok=True)
    os.makedirs(out_lbl, exist_ok=True)

    n = 0
    for fn in sorted(os.listdir(in_lbl)):
        if not fn.endswith(".json"):
            continue
        stem = fn[:-5]
        if not stem.startswith(cls + "_"):          # id 규칙: <class>_NNNNN
            continue
        if verified is not None and stem not in verified:
            continue
        with open(os.path.join(in_lbl, fn), encoding="utf-8") as f:
            label = json.load(f)
        label.pop("task", None)                     # 보조키 제거 → 순수 스키마만
        img = find_image(in_img, stem)
        if img is None:
            print(f"  [warn] 이미지 없음: {stem}");  continue
        shutil.copy2(img, os.path.join(out_img, os.path.basename(img)))
        with open(os.path.join(out_lbl, stem + ".json"), "w", encoding="utf-8") as f:
            json.dump(label, f, ensure_ascii=False, indent=2)
        n += 1
    return n


def print_cfg(cls, dst_root):
    reco = RECO.get(cls, {"image_size": [1280, 960], "max_length": 256})
    task = TASK_PROMPT[cls]
    tr = os.path.join(dst_root, "train").replace("\\", "/")
    va = os.path.join(dst_root, "val").replace("\\", "/")
    print("\n" + "=" * 70)
    print("아래를 donut_training.ipynb 의 Step 1 CFG 에 반영하세요 (로컬 모드):")
    print("=" * 70)
    print(f'''CFG["model"]["image_size"]   = {reco["image_size"]}   # 패치 크기에 맞춤(튜닝 가능)
CFG["model"]["max_length"]   = {reco["max_length"]}    # 라벨 JSON 이 짧아 작게
CFG["data"]["dataset_name"]  = None          # 로컬 모드
CFG["data"]["task_prompt"]   = "{task}"
CFG["data"]["local_train_dir"] = "{tr}"
CFG["data"]["local_val_dir"]   = "{va}"
# 학습 하이퍼(논문 P2 참고): num_epochs=30, learning_rate=3e-5, fp16=True
# 그리고 cell 9(로컬 데이터셋 준비)는 **건너뛰세요** — 이미 정리됨.''')
    print("=" * 70)

In [ ]:
def run_to_donut_vml(src, dvml, cls, verified_only=False):
    """donut_anno → donut_vml/data/processed/drawing_<class> 변환 + CFG 출력."""
    verified = load_verified(src) if verified_only else None
    if verified_only and verified is None:
        raise SystemExit(f"manifest.csv 없음: {src}")
    dst_root = os.path.join(dvml, "data", "processed", f"drawing_{cls}")
    n_tr = convert_split(src, "train", cls, dst_root, verified)
    n_va = convert_split(src, "val",   cls, dst_root, verified)
    print(f"[변환] class={cls}  train {n_tr} / val {n_va}  → {dst_root}")
    if n_tr == 0:
        raise SystemExit("학습 샘플 0개 — 클래스/검수/경로 확인")
    if n_va == 0:
        print("  [주의] val 0개 — split 의 VAL 비율을 늘리거나 데이터를 더 모으세요.")
    print_cfg(cls, dst_root)
    return dst_root

### 6-1. 실행 — 클래스 선택 후 변환
검수(`verified=1`)가 끝난 뒤 `VERIFIED_ONLY=True` 권장.

In [ ]:
SRC          = "datasets/donut_anno"
DVML         = "/home/jhkim/projects/donut_vml"
CLS          = "gdt"              # measure | radii | gdt | titleblock
VERIFIED_ONLY = False             # 검수 후 True 권장

run_to_donut_vml(SRC, DVML, CLS, verified_only=VERIFIED_ONLY)

## 다음 단계
1. 출력된 CFG 를 `donut_vml/donut_training.ipynb` Step 1 에 반영(로컬 모드).
2. 노트북 **cell 9(로컬 데이터 준비)는 건너뛴다**(이미 정리됨).
3. 실행 → `checkpoints/` 학습, Step 5 leaf-match 평가.

자세한 내용: `donut_data/README.md`.